# Positional Embedding

> **Absolute Positional embedding**
1.  Learned Absolute Positional Embeddings
2. Sinusoidal Positional Encoding (Original Transformer): https://www.youtube.com/watch?v=dWkm4nFikgM

> **Relative Positional embedding:**
1. Rotary Position Embeddings (RoPE):  https://www.youtube.com/watch?v=qKUobBR5R1A
2. Attention with Linear Biases (ALiBi): 
3. Axial Rotary Position Embeddings (for 2D/3D): 
4. 2D RoPE (for Vision Transformers): 

# Sinusoidal Positional Embedding

> **Why positional embeddings exist:** Attention computes relationships between all token pairs simultaneously — there is no sequential processing, no notion of "first" or "second." The sentence "dog bites man" and "man bites dog" produce identical attention inputs without positional information. Positional embeddings inject position into token representations before they enter the transformer.

---

## The Formula

For a token at position $pos$, its positional embedding is a vector of length $d_{\text{model}}$. Each dimension $i$ of this vector is computed as:

$$PE(pos,\ 2i)\ =\ \sin\!\left(\frac{pos}{10000^{\ 2i\,/\,d_{\text{model}}}}\right)$$

$$PE(pos,\ 2i+1)\ =\ \cos\!\left(\frac{pos}{10000^{\ 2i\,/\,d_{\text{model}}}}\right)$$

Even-indexed dimensions use sine, odd-indexed dimensions use cosine. Each dimension pair $i$ gets a different frequency: the divisor $10000^{2i/d_{\text{model}}}$ grows as $i$ increases, making later dimension pairs oscillate progressively more slowly.

---

## What Each Part Is Doing

Rewrite the argument as $pos \times \omega_i$ where:

$$\omega_i = \frac{1}{10000^{\ 2i\,/\,d_{\text{model}}}}$$

$\omega_i$ is the angular frequency for dimension pair $i$. For $d_{\text{model}} = 256$, here are the actual values at representative indices:

| Dimension pair $i$ | Exponent $2i/d_{\text{model}}$ | $\omega_i$ |
|---|---|---|
| 0 | 0.0 | 1.000000 — fastest |
| 32 | 0.25 | 0.100000 |
| 64 | 0.50 | 0.010000 |
| 96 | 0.75 | 0.001000 |
| 127 (max) | 0.992 | 0.000107 — slowest |

Early dimension pairs cycle through a full sine wave rapidly — their value changes significantly between adjacent positions. Later dimension pairs barely move — they change meaningfully only across thousands of positions.

This gives the embedding a **multiscale structure**:
- Early dimensions: fine-grained local ruler, sensitive to positions 1–10 apart
- Later dimensions: coarse global ruler, sensitive to positions hundreds to thousands apart

The slowest dimension ($i = 127$, $d_{\text{model}} = 256$) has period:

$$T_{\max} = \frac{2\pi}{\omega_{127}} = \frac{2\pi}{0.000107} \approx 58{,}470 \text{ positions}$$

This means the embedding provides meaningful positional variation across sequences of up to ~58,000 tokens before the slowest dimension completes one full cycle.

---

## Why Use Both Sine and Cosine — Is Sine Alone Not Enough?

This is the most commonly skipped explanation. The short answer: **sine alone breaks the linearity of positional shifts**, making it much harder for the model to learn relative position patterns.

### What we want

Ideally, if two tokens are always $k$ positions apart, the attention mechanism should detect this regardless of where they appear in the sequence. For attention to learn this, the operation "shift position by $k$" should correspond to a consistent linear transformation of the positional embedding — a matrix $M_k$ that depends only on $k$, not on $pos$:

$$PE(pos + k) = M_k \cdot PE(pos) \quad \text{for all } pos$$

If this holds, the model's Q and K weight matrices can learn to apply $M_k$ and detect fixed-distance relationships through their learned weights alone.

### Why sine alone cannot satisfy this

With sine only:

$$PE(pos,\ i) = \sin(pos \cdot \omega_i)$$

Can you express $\sin((pos + k)\omega_i)$ as a linear function of $\sin(pos \cdot \omega_i)$? Expanding with the angle addition formula:

$$\sin((pos + k)\omega_i) = \sin(pos\cdot\omega_i)\cos(k\omega_i)\ +\ \cos(pos\cdot\omega_i)\sin(k\omega_i)$$

The first term involves $\sin(pos \cdot \omega_i)$ — that is in our embedding. The second term involves $\cos(pos \cdot \omega_i)$ — **not** in our embedding if we only use sine. There is no way to express the shifted value as a linear combination of sine-only terms alone. The required information is absent, so no consistent $M_k$ exists.

### How adding cosine fixes this

With both sine and cosine for each dimension pair:

$$PE(pos,\ 2i) = \sin(pos \cdot \omega_i), \qquad PE(pos,\ 2i+1) = \cos(pos \cdot \omega_i)$$

Apply angle addition to both:

$$\sin((pos+k)\omega_i) = \sin(pos\cdot\omega_i)\cos(k\omega_i)\ +\ \cos(pos\cdot\omega_i)\sin(k\omega_i)$$

$$\cos((pos+k)\omega_i) = \cos(pos\cdot\omega_i)\cos(k\omega_i)\ -\ \sin(pos\cdot\omega_i)\sin(k\omega_i)$$

Both $\sin(pos \cdot \omega_i)$ and $\cos(pos \cdot \omega_i)$ are now available. Writing this as a matrix equation for dimension pair $i$:

$$\begin{bmatrix} \sin((pos+k)\omega_i) \\ \cos((pos+k)\omega_i) \end{bmatrix} = \begin{bmatrix} \cos(k\omega_i) & \sin(k\omega_i) \\ -\sin(k\omega_i) & \cos(k\omega_i) \end{bmatrix} \begin{bmatrix} \sin(pos\cdot\omega_i) \\ \cos(pos\cdot\omega_i) \end{bmatrix}$$

The $2 \times 2$ matrix is a **rotation matrix** — it rotates the (sin, cos) pair by angle $k \cdot \omega_i$. Every entry depends only on $k$ and $\omega_i$, not on $pos$ at all.

Stacking all dimension pairs, the full transformation $PE(pos) \to PE(pos+k)$ is a block-diagonal rotation matrix:

$$M_k = \begin{bmatrix} R(k\omega_0) & & & \\ & R(k\omega_1) & & \\ & & \ddots & \\ & & & R(k\omega_{d/2-1}) \end{bmatrix}$$

where each $R(\theta) = \begin{bmatrix} \cos\theta & \sin\theta \\ -\sin\theta & \cos\theta \end{bmatrix}$ is a $2\times 2$ rotation. This matrix depends only on $k$, so:

$$PE(pos + k) = M_k \cdot PE(pos) \quad \text{for all } pos \quad \checkmark$$

This is verified numerically: for $pos=17$, $k=5$, $d_{\text{model}}=8$, the maximum absolute difference between $M_k \cdot PE(pos)$ and the directly computed $PE(pos+k)$ is $1.1 \times 10^{-16}$ — pure floating-point rounding error.

### The practical consequence

The attention mechanism learns through Q and K weight matrices. When the model sees tokens consistently $k$ positions apart, it can learn the rotation $M_k$ through its weights — because $M_k$ is a fixed linear transformation. Without cosine, no such transformation exists, and the model must approximate positional reasoning through nonlinearities alone — a much harder learning problem.

---

## Why 10,000 as the Base

The base controls how spread out the frequency spectrum is. The following cosine similarity values are computed on $d_{\text{model}}=256$ embeddings.

### Too small a base (e.g. 100)

All dimensions oscillate fast. Similarity collapses very quickly:

| Pair | Distance | Similarity |
|---|---|---|
| sim(500, 510) | 10 | 0.36 |
| sim(500, 550) | 50 | 0.04 |
| sim(500, 600) | 100 | −0.07 |

Positions only 50 apart already look nearly unrelated. The model loses the ability to represent broad global structure — positions that should feel "close" look as unrelated as random vectors.

### Too large a base (e.g. 1,000,000)

All dimensions oscillate very slowly. Similarity barely changes across thousands of positions:

| Pair | Distance | Similarity |
|---|---|---|
| sim(0, 10) | 10 | 0.78 |
| sim(0, 100) | 100 | 0.61 |
| sim(0, 1000) | 1000 | 0.44 |
| sim(0, 5000) | 5000 | 0.30 |

Even positions 5,000 apart remain fairly similar (0.30). The model cannot distinguish "the adjacent word" from "a word 50 positions away" — local positional differences become invisible.

### Base = 10,000

Similarity decreases smoothly and gradually — neither collapsing immediately nor staying flat:

| Pair | Distance | Similarity |
|---|---|---|
| sim(500, 501) | 1 | 0.972 |
| sim(500, 510) | 10 | 0.675 |
| sim(500, 600) | 100 | 0.456 |
| sim(500, 1000) | 500 | 0.269 |
| sim(500, 2500) | 2000 | 0.048 |

Adjacent positions are very similar (0.97), moderate distances are noticeably different (0.46), and large distances become nearly unrelated (0.05). This smooth decay is ideal — the embedding simultaneously supports both local and global positional reasoning.

---

## Uniqueness Guarantee

Every position gets a distinct vector. Two positions share the same embedding only if all their (sin, cos) pairs are equal — which requires the same angle modulo $2\pi$ across every dimension simultaneously. Since each dimension pair has a different irrational frequency, this essentially never occurs within any practical context length.

---

## Properties Summary

| Property | Detail |
|---|---|
| Learned during training | No — fully deterministic |
| Unique per position | Yes |
| Extends beyond training length | Yes — formula works for any $pos$ |
| Encodes relative distances | Implicitly, via the rotation matrix property |
| Type | Absolute, fixed |

---
# Can Sinusoidal PE Handle Infinitely Long Sequences?

**Short answer:** Theoretically yes, practically no — bounded by two separate limits.

---

## Theoretically Unbounded

The formula accepts any integer $pos$ with no upper bound:

$$PE(pos,\ 2i) = \sin\!\left(\frac{pos}{10000^{\ 2i/d_{\text{model}}}}\right)$$

Plugging in $pos = 1{,}000{,}000$ produces a valid vector. The math never breaks.

---

## Practical Limit 1 — Uniqueness Breaks Down Past the Longest Period

- The slowest dimension (e.g. $i=127$ for $d_{\text{model}}=256$) has period $\approx 58{,}470$
- Once $pos$ exceeds this, that dimension starts repeating values
- Positions $pos$ and $pos + 58{,}470$ become identical in that dimension
- Other dimensions still differ, so full vectors stay unique for a while longer
- But as $pos$ grows further, more dimensions begin aliasing — distinct positions start looking increasingly similar to each other

> Uniqueness is practically guaranteed only up to roughly the longest period.

---

## Practical Limit 2 — Model Was Never Trained on Those Positions

- The model's weights were optimized on positions within the training context length
- Even if the formula produces a valid vector at $pos = 500{,}000$, the model has no learned behavior for it
- Attention patterns — how Q and K interact with those position vectors — were never trained for out-of-range positions
- Empirically, performance degrades noticeably once you exceed training context length

> A valid embedding is not the same as a useful one.

**Correct framing:** The formula is infinite. The uniqueness guarantee is finite. The model's ability to use those embeddings is bounded by training.


## The Remaining Limitation

Despite the rotation matrix property, sinusoidal embeddings are still **absolute** — each token's embedding depends only on its own position index. The matrix $M_k$ exists mathematically, but the model still has to discover and use it through learned Q and K weights. The distance $pos_i - pos_j$ is never directly computed and fed into the attention score — it must be inferred from two separate absolute position vectors.

This is the gap that **Relative Positional Embeddings** and **RoPE** address — encoding $pos_i - pos_j$ directly into the attention score computation, making relative position an explicit input rather than something to be inferred.

# Learnable Positional Embeddings

> **Builds on:** Sinusoidal positional embeddings
> **Problem it solves:** Sinusoidal embeddings are hand-designed with fixed frequencies. There is no reason the optimal positional representation for a given task should follow a mathematical formula. Learnable embeddings let the model discover what positional information is actually useful.

---

## The Core Idea

Instead of computing positional vectors with a formula, create a weight matrix of shape `[max_seq_len × d_model]` and treat each row as a learnable parameter. Each row is the positional embedding for that position index. These are randomly initialized and updated during training like any other model parameter.

```
P ∈ ℝ^{N × d_model}     # N = max sequence length

positional embedding for position pos = P[pos]   # just a row lookup
```

The input to the transformer becomes:

```
input = token_embedding[pos] + P[pos]
```

---

## How It Differs from Sinusoidal

| | Sinusoidal | Learnable |
|---|---|---|
| How computed | Fixed formula (sin/cos) | Looked up from trained matrix |
| Learned during training | No | Yes |
| Can adapt to task | No | Yes |
| Handles positions beyond training length | Yes (formula always works) | No (no embedding for unseen positions) |
| Type | Absolute, fixed | Absolute, learned |

---

## Who Uses It

Learnable positional embeddings became the default after the original transformer paper:

- **BERT** — learned embeddings up to position 512
- **GPT / GPT-2** — learned embeddings up to the context length
- **Vision Transformer (ViT)** — learned embeddings for image patch positions

The empirical finding across these models: learnable embeddings match or slightly outperform sinusoidal embeddings on most tasks, while being simpler to implement.

---

## The Fixed Maximum Length Problem

The most important limitation: the weight matrix has a fixed number of rows — `max_seq_len`. If the model sees a sequence longer than this at inference time, there is no positional embedding for those positions.

```
Trained with max_seq_len = 512
Input at inference: 600 tokens → positions 513–600 have no embedding
```

This is a hard constraint. Sinusoidal embeddings have no such limit — the formula works for any position. This difference becomes increasingly important as context lengths grow into the tens or hundreds of thousands of tokens.

---

## Why It Still Works Well Within Training Length

Within the trained range, the model has full freedom to shape each position's embedding however it finds useful. It may learn:

- Embeddings that cluster nearby positions together
- Embeddings that encode sentence boundaries
- Embeddings tuned to the specific token distribution of its training data

The learned structure often doesn't resemble sinusoidal waves at all — it reflects the statistical patterns the model encountered during training.

---

## Limitation: No Generalization to Relative Distances

Like sinusoidal embeddings, learnable positional embeddings are absolute — each embedding depends only on the position index, not on relationships between positions. If token A is always two positions before token B, the model must learn this pattern implicitly through attention weights, rather than having the distance encoded directly.

This is the key limitation that **Relative Positional Embeddings** and **RoPE** address — they encode the distance between tokens directly into the attention score computation, making relative position a first-class concept rather than something inferred from absolute positions.

# RoPE — Limitations

> **Assuming you know:** RoPE encodes relative position by rotating Q and K vectors by an angle proportional to their absolute position before computing attention scores. The dot product $Q_m \cdot K_n$ then depends only on the relative distance $m - n$, not on the absolute positions $m$ or $n$.

---

## Limitation 1 — Performance Degrades Beyond Training Context Length

RoPE rotation angles are $m \cdot \theta_i$ where $m$ is the absolute position. During training, the model sees positions up to some maximum $L$. The attention mechanism learns to interpret rotation angles in the range $[0,\ L \cdot \theta_i]$.

At inference, if $m > L$, the rotation angles fall outside the range seen during training. The model has no learned behavior for these angles — relative distances that involve out-of-range positions produce attention scores the model was never trained to interpret.

This is the same fundamental issue as sinusoidal PE, but it manifests differently: with sinusoidal PE the embeddings are added to inputs once, whereas with RoPE the out-of-range rotations directly corrupt the Q·K dot product at every layer.

In practice this means the model's performance drops sharply once the sequence exceeds the training context length — the model does not gracefully degrade, it fails.

---

## Limitation 2 — Long-Term Decay Is Not Always Monotonic

In theory, as relative distance $|m - n|$ increases, the expected dot product $Q_m \cdot K_n$ should decrease — tokens far apart should attend to each other less. RoPE achieves this on average through the cancellation of oscillating terms across dimensions.

However, this decay is not guaranteed to be monotonic for every specific pair of Q and K vectors. Depending on the learned weight matrices, certain long-range pairs can produce unexpectedly high attention scores — the oscillating nature of the rotation means there are distance values where dimensions constructively interfere rather than cancel.

This can cause the model to attend to irrelevant distant tokens more than intended, particularly for very long sequences.

---

## Limitation 3 — Incompatibility with Linear Attention

Standard attention is $O(N^2)$ in sequence length. Linear attention approximations reduce this to $O(N)$ by decomposing the softmax operation — but they require the ability to separate the Q and K computations into independent kernel features.

RoPE applies a position-dependent rotation to Q and K before the dot product:

$$Q_m \cdot K_n = (R_m q) \cdot (R_n k) = q^\top R_m^\top R_n k$$

The rotation $R_m^\top R_n$ couples positions $m$ and $n$ together. Linear attention requires a factorization of the form $f(Q_m) \cdot g(K_n)$ — but the coupling between $R_m$ and $R_n$ prevents this factorization. RoPE and linear attention are fundamentally incompatible without approximation.

---

## Limitation 4 — Fixed Base Frequency Limits Effective Context

The rotation angles use a fixed base (typically 10,000 — same as sinusoidal PE):

$$\theta_i = \frac{1}{10000^{\ 2i/d}}$$

For very long contexts, the high-frequency dimensions (small $i$) complete many full rotations over the sequence length. When a dimension completes a full $2\pi$ rotation, positions that are far apart produce the same rotation angle — the model cannot distinguish them in that dimension.

This frequency aliasing in high-frequency dimensions limits the effective range over which RoPE reliably encodes relative distance. It is one reason why simply training a RoPE model on longer sequences does not always improve long-context performance — the high-frequency dimensions alias before the model can learn from long-range dependencies.

This is what motivated **YaRN** and **RoPE scaling** techniques, which adjust the base frequency or rescale dimensions to extend the effective context range without full retraining.

---

## Limitation 5 — No Explicit Global Position Signal

RoPE encodes only relative distance between token pairs — it has no mechanism to signal absolute position to the model. In tasks where absolute position matters (e.g. "what is the first sentence?", structured documents with position-dependent semantics), the model must infer absolute position implicitly from the accumulated relative positions — which becomes increasingly unreliable for very long sequences.

---

## Limitation 6 — Incompatibility with KV Cache Compression (MLA)

As covered in MLA notes — RoPE applies a position-dependent rotation after the up-projection of K vectors. This breaks the matrix absorption trick that MLA relies on:

$$Q_i \cdot K_i^\top = (XW_{\downarrow}^Q W_{\uparrow,i}^Q R_m)\ \cdot\ (C^{KV} W_{\uparrow,i}^K R_n)^\top$$

The rotation matrices $R_m$ and $R_n$ sit between the matrices that MLA wants to merge into one. Since $R_m$ and $R_n$ are position-dependent (different for every token), they cannot be absorbed into a single static matrix — forcing the model to either recompute K for all positions at every step, or use the **Decoupled RoPE** workaround that maintains two separate Q/K paths.

---

## Summary

| Limitation | Root Cause |
|---|---|
| Degrades beyond training length | Angles outside trained range |
| Non-monotonic long-range decay | Oscillating dimensions can constructively interfere |
| Incompatible with linear attention | Position coupling prevents kernel factorization |
| High-frequency aliasing | Fixed base causes fast dimensions to wrap around |
| No absolute position signal | Purely relative by design |
| Breaks MLA absorption trick | Position-dependent rotation prevents matrix merging |

# RoPE Extensions

> **Builds on:** RoPE limitations
> **Problem they all solve:** RoPE's rotation angles $m \cdot \theta_i$ are calibrated to training context length $L$. At inference, positions $m > L$ produce angles the model has never seen — performance collapses sharply. All extensions below manipulate rotation angles to bring out-of-range positions back into, or near, the trained distribution without retraining from scratch.

---

## Quick Recap — What RoPE Actually Does

For position $m$ and dimension pair $i$, RoPE defines a base frequency:

$$\theta_i = \frac{1}{10000^{\ 2i/d}}$$

The Q and K vectors at position $m$ are rotated by angle $m \cdot \theta_i$ in each dimension pair. The dot product between $Q_m$ and $K_n$ then becomes:

$$Q_m \cdot K_n = f\!\left(m - n,\ \{\theta_i\}\right)$$

It depends only on the relative distance $m - n$, which is the whole point. But this only works correctly for distances the model has seen during training. Beyond training length $L$, the model encounters unfamiliar angle combinations and fails.

---

## The Core Tension All Extensions Must Navigate

Every extension must balance two competing needs:

- **Long-range coverage:** positions far apart must look sufficiently different from each other
- **Local sensitivity:** positions close together must also look different from each other

Changing frequencies in any direction hurts one of these. The approaches below differ in *how cleverly* they navigate this tradeoff.

---

## Approach 1 — Position Interpolation (PI)

**Source:** Chen et al., 2023 — *Extending Context Window of Large Language Models via Positional Interpolation*

### The idea

Instead of letting position indices grow beyond $L$, compress them so they always stay within $[0, L]$. If you want to support a new context length $L'$ where $L' > L$, replace every position $m$ with a scaled version $m'$:

$$m' = m \cdot \frac{L}{L'}$$

The modified RoPE rotation angle becomes:

$$\text{angle}_{m,i} = m' \cdot \theta_i = m \cdot \frac{L}{L'} \cdot \theta_i$$

### What this means concretely

Say your model was trained with $L = 4096$ and you want $L' = 8192$. The scaling factor is $4096/8192 = 0.5$. So:

- Position $m = 8192$ (the last token) maps to $m' = 4096$ — back inside the trained range
- Position $m = 4096$ maps to $m' = 2048$
- Position $m = 1$ maps to $m' = 0.5$

Every angle is now within the range the model saw during training. The model is no longer extrapolating — it is interpolating between known angle values.

### Why it works

The model has seen all angle values in $[0,\ L \cdot \theta_i]$ for every dimension during training. PI keeps every angle inside this range. No angle is ever out-of-distribution.

### The trade-off — local sensitivity degrades

Adjacent tokens at positions $m$ and $m+1$ now have angle difference:

$$\Delta\theta_i^{\text{PI}} = 1 \cdot \frac{L}{L'} \cdot \theta_i = \frac{L}{L'} \cdot \theta_i$$

Compare to the original:

$$\Delta\theta_i^{\text{original}} = 1 \cdot \theta_i$$

The angle difference between adjacent tokens is now $L/L'$ times smaller. For $L'/L = 2$, nearby tokens look twice as similar to each other as before. The model's ability to distinguish fine-grained local positions degrades proportionally.

**Fix:** A small amount of fine-tuning on long sequences after applying PI recovers most of this lost local sensitivity. The paper found that fine-tuning on as few as 1,000 steps was sufficient.

---

## Approach 2 — YaRN (Yet another RoPE extensioN)

**Source:** Peng et al., 2023 — *YaRN: Efficient Context Window Extension of Large Language Models*

### The problem with PI

PI scales all dimensions uniformly by $L/L'$. But this is wasteful and harmful for high-frequency dimensions.

Consider dimension $i = 0$: $\theta_0 = 1.0$, period $= 2\pi \approx 6.28$ positions. Adjacent tokens in this dimension are already at very different angles — there is no aliasing problem here even at long context. Scaling this dimension down makes nearby tokens look more similar than they should, degrading local sensitivity for no benefit.

Consider dimension $i = 127$ (for $d=256$): $\theta_{127} \approx 0.000107$, period $\approx 58{,}470$ positions. This dimension only completes one full cycle every 58,470 tokens — it aliases only for very long sequences. This is the dimension that actually needs rescaling.

**PI treats all dimensions the same. YaRN treats them differently based on their period.**

### The three-zone strategy

YaRN partitions dimensions into three groups based on their wavelength $\lambda_i = 2\pi / \theta_i$ relative to the original context length $L$ and target length $L'$:

**Zone 1 — High frequency** ($\lambda_i \ll L$, period much shorter than training length):
These dimensions already cycle many times within the training context. They have no aliasing problem. Leave them completely unchanged:

$$\text{scale}_i = 1 \quad \Rightarrow \quad \text{effective } \theta_i^{\text{new}} = \theta_i$$

**Zone 2 — Low frequency** ($\lambda_i \gg L'$, period much longer than target length):
These dimensions barely change across the entire sequence. Apply full interpolation — scale them by $L/L'$:

$$\text{scale}_i = \frac{L}{L'} \quad \Rightarrow \quad \text{effective } \theta_i^{\text{new}} = \frac{L}{L'} \cdot \theta_i$$

**Zone 3 — Middle frequency** (period between $L$ and $L'$):
Apply a smooth blend between the two extremes — a linear ramp from no scaling to full scaling as the period grows:

$$\text{scale}_i = \frac{1 - \frac{L}{L'}}{\lambda_i / (2\pi) - 1} \cdot \left(\frac{\lambda_i}{2\pi L} - 1\right) + \frac{L}{L'}$$

This is called **NTK-aware interpolation** — it borrows from Neural Tangent Kernel theory, which predicts that high-frequency components of a network's representation are more sensitive to perturbation and should be preserved.

### Temperature scaling

After rescaling, the interpolated embeddings have smaller magnitude differences between positions. This causes softmax attention to produce flatter distributions — the model attends more uniformly and less selectively. YaRN corrects this by scaling attention logits by a temperature factor $\sqrt{1/t}$ where $t > 1$:

$$\text{Attention} = \text{softmax}\!\left(\frac{Q_m K_n^\top \cdot \sqrt{1/t}}{\sqrt{d_k}}\right) V$$

The value of $t$ is determined empirically — typically around 0.1 to 0.3 log units above 1.

### Result

YaRN preserves local sensitivity far better than PI (high-frequency dimensions are untouched), requires minimal or no fine-tuning for moderate extensions, and achieves better perplexity on long documents than PI at the same extension ratio.

---

## Approach 3 — LongRoPE

**Source:** Ding et al., 2024 — *LongRoPE: Extending LLM Context Window Beyond 2 Million Tokens*

### The problem with YaRN

YaRN's three-zone strategy is hand-designed. The boundaries between zones and the blending formula are reasonable heuristics — but they are not guaranteed to be optimal for any specific model or context length.

LongRoPE asks: **what if we search for the optimal per-dimension rescaling factor rather than hand-designing it?**

### Evolutionary search for rescaling factors

LongRoPE treats the rescaling factor for each dimension as a parameter to optimize. It runs an evolutionary search (population-based optimization) to find a vector $\lambda \in \mathbb{R}^{d/2}$ where each $\lambda_i$ is the rescaling factor for dimension $i$:

$$\theta_i^{\text{new}} = \frac{\theta_i}{\lambda_i}$$

The search minimizes perplexity on a set of long documents. The result is a set of per-dimension rescaling factors that are specific to the model — not a general formula.

Different models with the same architecture but different training distributions end up with different optimal $\lambda$ vectors, which is precisely why a learned search outperforms a fixed formula.

### Special handling for boundary tokens

LongRoPE observes that position 0 (the first token) and the last token tend to receive disproportionately high attention across many heads — the model heavily uses them as anchors. Interpolating their positional embeddings distorts these anchor points and can destabilize attention patterns.

LongRoPE applies reduced or no rescaling to the very first and last positions, preserving their positional identity while still extending the range for all middle positions.

### Two-stage extension

For very long contexts (e.g. extending to 2M tokens), LongRoPE uses a two-stage process:
1. First extend to an intermediate length (e.g. 256K) using the searched $\lambda$ values with minimal fine-tuning
2. Then re-search for new $\lambda$ values calibrated to the final target length and fine-tune again

This staged approach avoids the degradation that occurs when trying to extend too far in one step.

### Result

Better perplexity than YaRN at very long contexts. Claims extension up to 2M tokens on LLaMA-based models with short fine-tuning. At the cost of requiring a model-specific optimization step.

---

## Approach 4 — Base Frequency Scaling

**Used in:** LLaMA 3 (base = 500,000), CodeLlama (base = 1,000,000), GPT-NeoX long context variants

### The idea

Instead of changing position indices or rescaling per-dimension, simply increase the base from 10,000 to a larger value $B'$:

$$\theta_i^{\text{new}} = \frac{1}{B'^{\ 2i/d}}$$

Since $B' > 10000$, every $\theta_i^{\text{new}} < \theta_i^{\text{original}}$ — all frequencies decrease, all periods lengthen.

### What this does to periods

For base $= 10000$: slowest period $\approx 58{,}470$ (for $d=256$)

For base $= 500{,}000$: slowest period:

$$T_{\max} = \frac{2\pi}{\theta_{127}} = \frac{2\pi}{1/500000^{0.992}} \approx \frac{2\pi}{0.0000023} \approx 2{,}700{,}000 \text{ positions}$$

The model can now meaningfully encode relative distances up to millions of tokens before aliasing occurs.

### The trade-off

Increasing the base slows down all dimensions. High-frequency dimensions (small $i$) that previously cycled every few tokens now cycle more slowly — adjacent tokens become harder to distinguish in those dimensions. Local sensitivity degrades, similar to PI but spread across all dimensions rather than uniformly compressing positions.

In practice, models trained from scratch with a large base (LLaMA 3 used 500,000) avoid this problem because the model learns to compensate during pretraining. The issue is more pronounced when switching base on a model pretrained with a smaller base — fine-tuning is required.

### Relationship to PI and YaRN

Increasing the base is mathematically equivalent to applying a non-uniform scaling to positions — different from PI (which scales positions uniformly) and from YaRN (which scales dimensions selectively). The equivalence:

$$\frac{m}{B'^{\ 2i/d}} = \frac{m}{10000^{\ 2i/d}} \cdot \left(\frac{10000}{B'}\right)^{2i/d}$$

The effective scaling factor $\left(\frac{10000}{B'}\right)^{2i/d}$ varies per dimension — larger for low-frequency dimensions (large $i$), smaller for high-frequency ones. This is a form of non-uniform dimension-dependent scaling, but derived from a single scalar $B'$ rather than per-dimension optimization.

---

## Comparison

| Method | What changes | Local sensitivity | Long-range coverage | Needs fine-tuning | Complexity |
|---|---|---|---|---|---|
| Position Interpolation | Scales all positions by $L/L'$ | Degrades uniformly | Good up to $L'$ | Yes, a little | Simple |
| YaRN | Scales dimensions selectively | Mostly preserved | Good | Minimal | Moderate |
| LongRoPE | Optimizes per-dimension $\lambda_i$ | Best preserved | Best | Minimal | High (requires search) |
| Base frequency scaling | Increases base $B$ | Slight degradation | Very long range | Yes (if post-hoc) | Simple |

---

## The Common Pattern

Every approach is a manipulation of the angle $m \cdot \theta_i$. They differ only in *what they modify*:

```
Original RoPE:             angle = m          ×  θᵢ
Position Interpolation:    angle = m·(L/L')   ×  θᵢ          (scale m)
YaRN:                      angle = m          ×  θᵢ·scaleᵢ   (scale θ selectively)
LongRoPE:                  angle = m          ×  θᵢ/λᵢ        (optimize scale per dim)
Base frequency scaling:    angle = m          ×  1/B'^(2i/d)  (change base)
```

All of them try to ensure that for any two positions $m$ and $n$ within the new context length $L'$, the angle combination $(m \cdot \theta_i^{\text{new}},\ n \cdot \theta_i^{\text{new}})$ resembles something the model saw during its original training on $[0, L]$.

---

> **What none of them fully solve:** These are all approximations. The model's weights were optimized for a specific angle distribution — any extension introduces some distribution shift. The cleanest solution remains training with long contexts from the start, which is why modern models like LLaMA 3 use a large base (500,000) from pretraining rather than applying extensions post-hoc.